# Price forecasting model

Gradient-boosting day-ahead price forecast with leakage-safe lag and calendar features.


## Design choices

The model uses calendar features and price values observed before the price being predicted:

- Hour-of-day and day-of-week cycles
- Price lags of 24, 48, 72, 168 and 336 hours
- Rolling 24-hour mean and standard deviation, shifted by one hour

Every lag and rolling statistic uses earlier observations only. The target price at time \(t\) is never included in its own feature vector. Evaluation remains chronological: train on 2015–2022, select on 2023 validation, and report once on the untouched 2024 test period.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

DATA_PATH = Path("data/belgium_day_ahead_hourly_2015_2024.csv")
FORECAST_OUTPUT_PATH = Path("data/forecast_test_2024.csv")

prices = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])
prices["timestamp"] = pd.to_datetime(prices["timestamp"], utc=True, errors="raise")
prices = prices.sort_values("timestamp").reset_index(drop=True)

assert prices["timestamp"].is_monotonic_increasing
assert not prices["timestamp"].duplicated().any()
assert prices["price_eur_per_mwh"].notna().all()
prices.head()


,timestamp,price_eur_per_mwh
0,2015-01-04 23:00:00+00:00,36.56
1,2015-01-05 00:00:00+00:00,34.94
2,2015-01-05 01:00:00+00:00,32.19
3,2015-01-05 02:00:00+00:00,28.05
4,2015-01-05 03:00:00+00:00,28.04


## Create features without leakage


In [2]:
def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Return calendar and historical-price features available before each target hour."""
    featured = frame.copy()

    hour = featured["timestamp"].dt.hour
    day_of_week = featured["timestamp"].dt.dayofweek
    featured["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    featured["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    featured["day_of_week_sin"] = np.sin(2 * np.pi * day_of_week / 7)
    featured["day_of_week_cos"] = np.cos(2 * np.pi * day_of_week / 7)

    for lag_hours in [24, 48, 72, 168, 336]:
        featured[f"lag_{lag_hours}h"] = featured["price_eur_per_mwh"].shift(lag_hours)

    # Shift first: the target hour's price must not enter its own rolling statistic.
    historical_price = featured["price_eur_per_mwh"].shift(1)
    featured["rolling_mean_24h"] = historical_price.rolling(24).mean()
    featured["rolling_std_24h"] = historical_price.rolling(24).std()

    return featured

featured_prices = add_features(prices)

FEATURE_COLUMNS = [
    "hour_sin",
    "hour_cos",
    "day_of_week_sin",
    "day_of_week_cos",
    "lag_24h",
    "lag_48h",
    "lag_72h",
    "lag_168h",
    "lag_336h",
    "rolling_mean_24h",
    "rolling_std_24h",
]

featured_prices.loc[330:340, ["timestamp", "price_eur_per_mwh", *FEATURE_COLUMNS]].head()


,timestamp,price_eur_per_mwh,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos,lag_24h,lag_48h,lag_72h,lag_168h,lag_336h,rolling_mean_24h,rolling_std_24h
330,2015-01-18 17:00:00+00:00,53.67,-0.965926,-2.588190e-01,-0.781831,0.62349,45.84,53.54,50.45,50.00,NaN,36.332917,6.601093
331,2015-01-18 18:00:00+00:00,51.02,-1.000000,-1.836970e-16,-0.781831,0.62349,46.43,48.02,49.47,42.00,NaN,36.659167,7.252727
332,2015-01-18 19:00:00+00:00,50.62,-0.965926,2.588190e-01,-0.781831,0.62349,39.96,41.51,42.98,36.08,NaN,36.850417,7.574941
333,2015-01-18 20:00:00+00:00,42.44,-0.866025,5.000000e-01,-0.781831,0.62349,32.60,38.66,38.00,34.06,NaN,37.294583,8.062071
334,2015-01-18 21:00:00+00:00,44.94,-0.707107,7.071068e-01,-0.781831,0.62349,37.78,40.64,35.65,33.00,NaN,37.704583,8.063155


## Chronological train, validation and test sets


In [3]:
TRAIN_END = pd.Timestamp("2023-01-01", tz="UTC")
VALIDATION_END = pd.Timestamp("2024-01-01", tz="UTC")
TEST_END = pd.Timestamp("2025-01-01", tz="UTC")

model_ready = featured_prices.dropna(subset=FEATURE_COLUMNS).copy()

train = model_ready.loc[model_ready["timestamp"] < TRAIN_END].copy()
validation = model_ready.loc[
    (model_ready["timestamp"] >= TRAIN_END)
    & (model_ready["timestamp"] < VALIDATION_END)
].copy()
test = model_ready.loc[
    (model_ready["timestamp"] >= VALIDATION_END)
    & (model_ready["timestamp"] < TEST_END)
].copy()

assert train["timestamp"].max() < validation["timestamp"].min()
assert validation["timestamp"].max() < test["timestamp"].min()
assert len(validation) == 365 * 24
assert len(test) == 366 * 24

pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(train), len(validation), len(test)],
        "start": [train["timestamp"].min(), validation["timestamp"].min(), test["timestamp"].min()],
        "end": [train["timestamp"].max(), validation["timestamp"].max(), test["timestamp"].max()],
    }
)


,split,rows,start,end
0,train,69697,2015-01-18 23:00:00+00:00,2022-12-31 23:00:00+00:00
1,validation,8760,2023-01-01 00:00:00+00:00,2023-12-31 23:00:00+00:00
2,test,8784,2024-01-01 00:00:00+00:00,2024-12-31 23:00:00+00:00


## Train one model


In [4]:
model = HistGradientBoostingRegressor(
    loss="absolute_error",
    max_iter=300,
    learning_rate=0.06,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42,
)

model.fit(train[FEATURE_COLUMNS], train["price_eur_per_mwh"])
print(f"Trained on {len(train):,} hourly observations.")


Trained on 69,697 hourly observations.


## Compare with the benchmark


In [5]:
def score_forecast(actual: pd.Series, prediction: np.ndarray | pd.Series) -> dict:
    """Calculate point-forecast error metrics in EUR/MWh."""
    return {
        "mae_eur_per_mwh": mean_absolute_error(actual, prediction),
        "rmse_eur_per_mwh": mean_squared_error(actual, prediction) ** 0.5,
    }

def evaluate_period(name: str, frame: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray]:
    """Score the ML model and same-hour-yesterday benchmark on one fixed period."""
    model_prediction = model.predict(frame[FEATURE_COLUMNS])
    yesterday_prediction = frame["lag_24h"]

    results = pd.DataFrame(
        [
            {"period": name, "method": "Histogram Gradient Boosting", **score_forecast(frame["price_eur_per_mwh"], model_prediction)},
            {"period": name, "method": "Same hour yesterday", **score_forecast(frame["price_eur_per_mwh"], yesterday_prediction)},
        ]
    )
    return results, model_prediction

validation_results, validation_prediction = evaluate_period("validation_2023", validation)
test_results, test_prediction = evaluate_period("test_2024", test)

forecast_results = pd.concat([validation_results, test_results], ignore_index=True)
forecast_results[["mae_eur_per_mwh", "rmse_eur_per_mwh"]] = forecast_results[
    ["mae_eur_per_mwh", "rmse_eur_per_mwh"]
].round(2)
forecast_results


,period,method,mae_eur_per_mwh,rmse_eur_per_mwh
0,validation_2023,Histogram Gradient Boosting,17.05,23.68
1,validation_2023,Same hour yesterday,23.76,34.56
2,test_2024,Histogram Gradient Boosting,17.18,23.95
3,test_2024,Same hour yesterday,23.19,33.24


## Preserve final-test forecasts


In [6]:
test_forecasts = test[["timestamp", "price_eur_per_mwh", "lag_24h"]].copy()
test_forecasts = test_forecasts.rename(
    columns={"lag_24h": "forecast_yesterday_eur_per_mwh"}
)
test_forecasts["forecast_model_eur_per_mwh"] = test_prediction

test_forecasts.to_csv(FORECAST_OUTPUT_PATH, index=False)

assert len(test_forecasts) == len(test)
assert test_forecasts[["forecast_yesterday_eur_per_mwh", "forecast_model_eur_per_mwh"]].notna().all().all()

print(f"Saved {len(test_forecasts):,} final-test forecasts to {FORECAST_OUTPUT_PATH}")
test_forecasts.head()


Saved 8,784 final-test forecasts to data/forecast_test_2024.csv


,timestamp,price_eur_per_mwh,forecast_yesterday_eur_per_mwh,forecast_model_eur_per_mwh
78793,2024-01-01 00:00:00+00:00,0.01,8.33,10.381210
78794,2024-01-01 01:00:00+00:00,0.00,4.00,5.930276
78795,2024-01-01 02:00:00+00:00,-0.01,0.31,6.111606
78796,2024-01-01 03:00:00+00:00,-0.03,-0.06,6.388514
78797,2024-01-01 04:00:00+00:00,-0.02,-0.04,7.209996
